# MedVision-AI Kaggle GPU Training Notebook

This notebook orchestrates Stage 1 and Stage 2 training for MedVision-AI on Kaggle.

## Workflow:
1. **Env Setup**: Verify Python, TensorFlow/Keras, GPU, and canonical paths
2. **Repo Sync**: Clone or update from GitHub; preserve artifacts
3. **Validation**: Distinguish Stage 1 resume vs Stage 2 source checkpoint validation
4. **Stage 2 Forensic** (Optional): FP32 vs mixed_float16 numerical comparison (10 batches max)
5. **Stage 1 Launcher** (Optional): Only if no valid Stage 1 checkpoint exists
6. **Stage 2 Launcher** (Active): Production fine-tuning with auto-resume
7. **Stage 2 Checkpoint Verification**: Confirm persistence
8. **Artifact Inventory**: List all outputs
9. **Final ZIP**: Package artifacts
10. **Download Link**: Get artifacts


In [5]:
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/MedVision-AI")

commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Repository:", repo)
print("Git HEAD:", commit)

Repository: /kaggle/working/MedVision-AI
Git HEAD: 515c63e


In [6]:
from pathlib import Path

root = Path("/kaggle/input")

print("=== Candidate model files ===")

for p in root.rglob("*"):
    if p.is_file() and (
        p.name.endswith(".keras")
        or p.name.endswith(".zip")
        or p.name.endswith(".h5")
    ):
        print(p)

=== Candidate model files ===
/kaggle/input/datasets/swastikpandey004/stage2-file/densenet121_stage2_best.keras
/kaggle/input/datasets/swastikpandey004/stage1-folder/medvision_stage1_artifacts/checkpoints/densenet121_stage1_best.keras


In [7]:
from pathlib import Path
import shutil

input_root = Path("/kaggle/input")

candidates = [
    p for p in input_root.rglob("densenet121_stage2_best.keras")
    if p.is_file()
]

print("Stage 2 checkpoint candidates:")
for p in candidates:
    print(" ", p)

if len(candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one densenet121_stage2_best.keras, "
        f"found {len(candidates)}"
    )

source = candidates[0]

destination = (
    Path("/kaggle/working/medvision_outputs")
    / "checkpoints"
    / "densenet121_stage2_best.keras"
)

destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, destination)

print("\nCopied successfully:")
print("Source      :", source)
print("Destination :", destination)
print("Size MB     :", round(destination.stat().st_size / 1024**2, 2))

Stage 2 checkpoint candidates:
  /kaggle/input/datasets/swastikpandey004/stage2-file/densenet121_stage2_best.keras

Copied successfully:
Source      : /kaggle/input/datasets/swastikpandey004/stage2-file/densenet121_stage2_best.keras
Destination : /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras
Size MB     : 38.25


In [8]:
from pathlib import Path

checkpoint = Path(
    "/kaggle/working/medvision_outputs/checkpoints/"
    "densenet121_stage2_best.keras"
)

print("=" * 70)
print("STAGE 2 CHECKPOINT")
print("=" * 70)
print("Exists :", checkpoint.exists())

if checkpoint.exists():
    print("Size MB:", round(checkpoint.stat().st_size / 1024**2, 2))
    print("Path   :", checkpoint)

STAGE 2 CHECKPOINT
Exists : True
Size MB: 38.25
Path   : /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras


In [9]:
import keras

model = keras.models.load_model(
    checkpoint,
    compile=False,
)

print("MODEL LOAD: PASS")
print("Model type:", type(model).__name__)
print("Parameters:", model.count_params())

I0000 00:00:1787364458.511645      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787364458.514587      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


MODEL LOAD: PASS
Model type: Functional
Parameters: 7301185


In [10]:
from pathlib import Path

dataset = Path(
    "/kaggle/input/competitions/"
    "rsna-pneumonia-detection-challenge"
)

print("Dataset exists:", dataset.exists())

labels = list(dataset.glob("*train_labels.csv"))
print("Label files:", labels)

Dataset exists: True
Label files: [PosixPath('/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv')]


In [11]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/MedVision-AI")
checkpoint = Path(
    "/kaggle/working/medvision_outputs/checkpoints/"
    "densenet121_stage2_best.keras"
)

output_dir = Path(
    "/kaggle/working/medvision_outputs/evaluation"
)

output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "scripts/evaluate.py",
    "--checkpoint", str(checkpoint),
    "--split", "val",
    "--mode", "full",
    "--batch-size", "32",
    "--output-dir", str(output_dir),
]

print("Running:")
print(" ".join(cmd))

subprocess.run(
    cmd,
    cwd=repo,
    check=True,
)

Running:
/usr/bin/python3 scripts/evaluate.py --checkpoint /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras --split val --mode full --batch-size 32 --output-dir /kaggle/working/medvision_outputs/evaluation
[2026-08-22 02:08:44] [INFO] [medvision.evaluate_script:230] - Loading model checkpoint from: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras


I0000 00:00:1787364525.458268     226 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13590 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787364525.460515     226 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13654 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[2026-08-22 02:08:49] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-22 02:08:49] [INFO] [medvision.evaluate_script:95] - Resolved evaluation data mode: 'full' (requested: 'full')
[2026-08-22 02:08:49] [INFO] [medvision.evaluate_script:114] - Building official patient-aware 70/15/15 splits from RSNA dataset at /kaggle/input/competitions/rsna-pneumonia-detection-challenge...
[2026-08-22 02:09:32] [INFO] [medvision.data:144] - Manifest created successfully: 26684 unique patient records parsed.
[2026-08-22 02:09:33] [INFO] [medvision.data.splits:46] - Patient Leakage Audit PASSED: 0% patient overlap across all splits.
[2026-08-22 02:09:33] [INFO] [medvision.data.splits:46] - Patient Leakage Audit PASSED: 0% patient overlap across all splits.
[2026-08-22 02:09:33] [INFO] [medvision.data.splits:110] - Splits created cleanly: Train=18678 (70.0%), Val=4003 (15.0%), Test=4003 (15.0%)
[2026-08-22 02:09:33] [INFO

CompletedProcess(args=['/usr/bin/python3', 'scripts/evaluate.py', '--checkpoint', '/kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras', '--split', 'val', '--mode', 'full', '--batch-size', '32', '--output-dir', '/kaggle/working/medvision_outputs/evaluation'], returncode=0)

In [12]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/MedVision-AI")
checkpoint = Path(
    "/kaggle/working/medvision_outputs/checkpoints/"
    "densenet121_stage2_best.keras"
)

output_dir = Path(
    "/kaggle/working/medvision_outputs/evaluation"
)
output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "scripts/evaluate.py",
    "--checkpoint", str(checkpoint),
    "--split", "test",
    "--mode", "full",
    "--batch-size", "32",
    "--output-dir", str(output_dir),
]

print("Running:")
print(" ".join(cmd))

subprocess.run(
    cmd,
    cwd=repo,
    check=True,
)

Running:
/usr/bin/python3 scripts/evaluate.py --checkpoint /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras --split test --mode full --batch-size 32 --output-dir /kaggle/working/medvision_outputs/evaluation
[2026-08-22 02:14:44] [INFO] [medvision.evaluate_script:230] - Loading model checkpoint from: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras


I0000 00:00:1787364885.095293     307 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13590 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787364885.097630     307 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13654 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[2026-08-22 02:14:48] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-22 02:14:48] [INFO] [medvision.evaluate_script:95] - Resolved evaluation data mode: 'full' (requested: 'full')
[2026-08-22 02:14:48] [INFO] [medvision.evaluate_script:114] - Building official patient-aware 70/15/15 splits from RSNA dataset at /kaggle/input/competitions/rsna-pneumonia-detection-challenge...
[2026-08-22 02:14:52] [INFO] [medvision.data:144] - Manifest created successfully: 26684 unique patient records parsed.
[2026-08-22 02:14:52] [INFO] [medvision.data.splits:46] - Patient Leakage Audit PASSED: 0% patient overlap across all splits.
[2026-08-22 02:14:52] [INFO] [medvision.data.splits:46] - Patient Leakage Audit PASSED: 0% patient overlap across all splits.
[2026-08-22 02:14:52] [INFO] [medvision.data.splits:110] - Splits created cleanly: Train=18678 (70.0%), Val=4003 (15.0%), Test=4003 (15.0%)
[2026-08-22 02:14:52] [INFO

CompletedProcess(args=['/usr/bin/python3', 'scripts/evaluate.py', '--checkpoint', '/kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras', '--split', 'test', '--mode', 'full', '--batch-size', '32', '--output-dir', '/kaggle/working/medvision_outputs/evaluation'], returncode=0)

In [13]:
from pathlib import Path
import shutil

evaluation_dir = Path("/kaggle/working/medvision_outputs/evaluation")
zip_base = Path("/kaggle/working/MedVision-AI-full-evaluation")

print("Evaluation directory exists:", evaluation_dir.exists())

if not evaluation_dir.exists():
    raise FileNotFoundError(
        f"Evaluation directory not found: {evaluation_dir}"
    )

files = [p for p in evaluation_dir.rglob("*") if p.is_file()]

print(f"Files found: {len(files)}")
for p in files:
    print(" -", p.relative_to(evaluation_dir))

archive_path = Path(
    shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=evaluation_dir,
    )
)

print("\nZIP created:")
print(archive_path)
print("Size MB:", round(archive_path.stat().st_size / 1024**2, 2))

Evaluation directory exists: True
Files found: 16
 - densenet121_stage2_best_val_confusion_matrix.svg
 - densenet121_stage2_best_test_pr_curve.png
 - densenet121_stage2_best_val_roc_curve.svg
 - densenet121_stage2_best_test_confusion_matrix.png
 - densenet121_stage2_best_test_roc_curve.svg
 - densenet121_stage2_best_test_report.md
 - densenet121_stage2_best_val_confusion_matrix.png
 - densenet121_stage2_best_test_report.json
 - densenet121_stage2_best_val_report.md
 - densenet121_stage2_best_val_roc_curve.png
 - densenet121_stage2_best_test_roc_curve.png
 - densenet121_stage2_best_test_pr_curve.svg
 - densenet121_stage2_best_val_report.json
 - densenet121_stage2_best_val_pr_curve.svg
 - densenet121_stage2_best_test_confusion_matrix.svg
 - densenet121_stage2_best_val_pr_curve.png

ZIP created:
/kaggle/working/MedVision-AI-full-evaluation.zip
Size MB: 0.45


In [15]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/MedVision-AI")

print("Repository:", repo)

subprocess.run(
    ["git", "-C", str(repo), "fetch", "origin"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(repo), "reset", "--hard", "origin/main"],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(repo), "--no-deps"],
    check=True,
)

commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Kaggle repository HEAD:", commit)

Repository: /kaggle/working/MedVision-AI


From https://github.com/SwastikPandey1024/MedVision-AI
   515c63e..ed358be  main       -> origin/main


HEAD is now at ed358be feat(evaluation): add validation-only threshold selection and final test reporting
Obtaining file:///kaggle/working/MedVision-AI
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for medvision-ai (pyproject.toml): started
  Building editable for medvision-ai (pyproject.toml): finished with status 'done'
  Created wheel for medvision-ai: filename=medvision_ai-0.1.0a0-0.editable-py3-none-any.whl size=5465 sha256=634f3446fd9c5d4c1f284de28864f32b8e8209e8c0d3bc7f6af2f99b838fa799
  Stored in directory: /tmp/pip-

In [16]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/MedVision-AI")

result = subprocess.run(
    [sys.executable, "scripts/evaluate.py", "--help"],
    cwd=repo,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

usage: evaluate.py [-h] --checkpoint CHECKPOINT
                   [--mode {auto,full,development}]
                   [--split {val,test,dev,all}] [--batch-size BATCH_SIZE]
                   [--threshold THRESHOLD] [--optimize-threshold]
                   [--threshold-criterion {f1_score,accuracy,youden_j}]
                   [--output-dir OUTPUT_DIR] [--dataset-dir DATASET_DIR]

MedVision-AI Model Evaluation CLI

options:
  -h, --help            show this help message and exit
  --checkpoint CHECKPOINT
                        Path to trained .keras model checkpoint file.
  --mode {auto,full,development}
                        Dataset resolution mode ('auto', 'full',
                        'development'). Default: 'auto'.
  --split {val,test,dev,all}
                        Dataset split to evaluate ('val', 'test', 'dev',
                        'all'). Default: 'val'.
  --batch-size BATCH_SIZE
                        Batch size for dataset iteration (default: 32).
  --threshold T

In [17]:
from pathlib import Path

checkpoint = Path(
    "/kaggle/working/medvision_outputs/checkpoints/"
    "densenet121_stage2_best.keras"
)

print("Exists:", checkpoint.exists())

if checkpoint.exists():
    print("Size MB:", round(checkpoint.stat().st_size / 1024**2, 2))

Exists: True
Size MB: 38.25


In [18]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/MedVision-AI")

checkpoint = Path(
    "/kaggle/working/medvision_outputs/checkpoints/"
    "densenet121_stage2_best.keras"
)

output_dir = Path(
    "/kaggle/working/medvision_outputs/evaluation_threshold"
)

output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "scripts/evaluate.py",
    "--checkpoint", str(checkpoint),
    "--split", "all",
    "--mode", "full",
    "--batch-size", "32",
    "--optimize-threshold",
    "--threshold-criterion", "f1_score",
    "--output-dir", str(output_dir),
]

print("Running:")
print(" ".join(cmd))

subprocess.run(
    cmd,
    cwd=repo,
    check=True,
)

Running:
/usr/bin/python3 scripts/evaluate.py --checkpoint /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras --split all --mode full --batch-size 32 --optimize-threshold --threshold-criterion f1_score --output-dir /kaggle/working/medvision_outputs/evaluation_threshold
[2026-08-22 02:45:11] [INFO] [medvision.evaluate_script:245] - Loading model checkpoint from: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras


I0000 00:00:1787366711.936813     472 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13590 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787366711.939056     472 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13654 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[2026-08-22 02:45:15] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-22 02:45:15] [INFO] [medvision.evaluate_script:97] - Resolved evaluation data mode: 'full' (requested: 'full')
[2026-08-22 02:45:15] [INFO] [medvision.evaluate_script:116] - Building official patient-aware 70/15/15 splits from RSNA dataset at /kaggle/input/competitions/rsna-pneumonia-detection-challenge...
[2026-08-22 02:45:19] [INFO] [medvision.data:144] - Manifest created successfully: 26684 unique patient records parsed.
[2026-08-22 02:45:19] [INFO] [medvision.data.splits:46] - Patient Leakage Audit PASSED: 0% patient overlap across all splits.
[2026-08-22 02:45:19] [INFO] [medvision.data.splits:46] - Patient Leakage Audit PASSED: 0% patient overlap across all splits.
[2026-08-22 02:45:19] [INFO] [medvision.data.splits:110] - Splits created cleanly: Train=18678 (70.0%), Val=4003 (15.0%), Test=4003 (15.0%)
[2026-08-22 02:45:19] [INFO

CompletedProcess(args=['/usr/bin/python3', 'scripts/evaluate.py', '--checkpoint', '/kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras', '--split', 'all', '--mode', 'full', '--batch-size', '32', '--optimize-threshold', '--threshold-criterion', 'f1_score', '--output-dir', '/kaggle/working/medvision_outputs/evaluation_threshold'], returncode=0)

In [1]:
import os
import platform
import sys
from pathlib import Path
import subprocess

# Set Keras backend BEFORE any TensorFlow/Keras imports
os.environ['KERAS_BACKEND'] = 'tensorflow'

import tensorflow as tf
import keras

# Print environment
print('=' * 80)
print('ENVIRONMENT')
print('=' * 80)
print(f'Python version: {platform.python_version()}')
print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version: {keras.__version__}')
print(f'GPU devices: {len(tf.config.list_physical_devices("GPU"))}')
for gpu in tf.config.list_physical_devices('GPU'):
    print(f'  - {gpu}')
print(f'KERAS_BACKEND: {os.environ.get("KERAS_BACKEND")}')

# Canonical paths
REPO = Path('/kaggle/working') / 'MedVision-AI'
OUTPUTS = Path('/kaggle/working') / 'medvision_outputs'
CHECKPOINTS = OUTPUTS / 'checkpoints'

print(f'\nCanonical paths:')
print(f'  REPO: {REPO}')
print(f'  OUTPUTS: {OUTPUTS}')
print(f'  CHECKPOINTS: {CHECKPOINTS}')
print('=' * 80)


ENVIRONMENT
Python version: 3.12.13
TensorFlow version: 2.20.0
Keras version: 3.13.2
GPU devices: 2
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
  - PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
KERAS_BACKEND: tensorflow

Canonical paths:
  REPO: /kaggle/working/MedVision-AI
  OUTPUTS: /kaggle/working/medvision_outputs
  CHECKPOINTS: /kaggle/working/medvision_outputs/checkpoints


In [2]:
# Clone or update repository (preserve artifacts)
if REPO.exists():
    print(f'Repository already cloned at {REPO}')
    os.chdir(str(REPO))
    result = subprocess.run(['git', 'fetch', 'origin'], capture_output=True, text=True)
    print('git fetch origin:', result.stdout.strip() if result.stdout else '(up to date)')
    result = subprocess.run(['git', 'reset', '--hard', 'origin/main'], capture_output=True, text=True)
    print('git reset --hard origin/main:', result.stdout.strip())
else:
    print(f'Cloning repository to {REPO}...')
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', 'https://github.com/SwastikPandey1024/MedVision-AI.git', str(REPO)],
        check=True
    )
    os.chdir(str(REPO))

# Install package in development mode (no dependencies, we have TensorFlow from Kaggle)
print('\nInstalling package: pip install -e . --no-deps')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps', '-q'], check=True)

# Print exact commit
result = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, cwd=str(REPO))
commit_sha = result.stdout.strip()
print(f'\nRepository at commit: {commit_sha}')
print('=' * 80)


Cloning repository to /kaggle/working/MedVision-AI...


Cloning into '/kaggle/working/MedVision-AI'...



Installing package: pip install -e . --no-deps

Repository at commit: 515c63e9114c8ebf2bfbe0a64b18bcd776806f56


In [3]:
import sys
from pathlib import Path

REPO = Path("/kaggle/working/MedVision-AI")
SRC = REPO / "src"

print("REPO exists:", REPO.exists())
print("SRC exists:", SRC.exists())

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import medvision

print("✅ MedVision import OK")
print("Path:", medvision.__file__)
print("SRC in sys.path:", str(SRC) in sys.path)

REPO exists: True
SRC exists: True
✅ MedVision import OK
Path: /kaggle/working/MedVision-AI/src/medvision/__init__.py
SRC in sys.path: True


In [4]:
from medvision.config.settings import get_output_dir
from medvision.data.dataset import find_dataset_root, parse_rsna_manifest, create_real_rsna_dataset
from medvision.data.splits import create_patient_aware_splits
from medvision.models.trainer import (
    find_valid_resume_checkpoint,
    validate_stage2_source_checkpoint,
    resolve_stage2_source_checkpoint,
)

print('=' * 80)
print('VALIDATION: DATASET & CHECKPOINT PROVENANCE')
print('=' * 80)

# Find dataset
dataset_root = find_dataset_root()
print(f'Dataset root: {dataset_root}')
if dataset_root:
    manifest_path = dataset_root / 'manifest.csv'
    if manifest_path.exists():
        manifest = parse_rsna_manifest(str(manifest_path))
        print(f'RSNA manifest: {len(manifest)} images')
    else:
        print('Warning: manifest.csv not found in dataset root')
else:
    print('Error: RSNA dataset not found. Ensure Kaggle RSNA attachment is available.')

# Stage 1 Resume Checkpoint Validation
# Requires: optimizer state, exact epoch recovery, weights match, architecture match
print('\n' + '=' * 80)
print('STAGE 1 RESUME VALIDATION (Optimizer State Required)')
print('=' * 80)
stage1_ckpt = get_output_dir('checkpoints') / 'densenet121_stage1_best.keras'
try:
    result = find_valid_resume_checkpoint(str(stage1_ckpt), 'densenet121')
    print(f'Stage 1 checkpoint: {stage1_ckpt.name}')
    print(f'  Status: {result.status}')
    print(f'  Reason: {result.reason}')
except Exception as e:
    print(f'Stage 1 checkpoint validation: {e}')

# Stage 2 Source Checkpoint Validation
# Requires: model weights & architecture only (no optimizer state needed)
print('\n' + '=' * 80)
print('STAGE 2 SOURCE VALIDATION (Model Weights Only, Optimizer NOT Required)')
print('=' * 80)
stage2_src_ckpt = get_output_dir('checkpoints') / 'densenet121_stage1_best.keras'
try:
    source_result = resolve_stage2_source_checkpoint(str(stage1_ckpt), None, 'densenet121')
    print(f'Stage 2 source: {source_result.checkpoint_path}')
    print(f'  Status: {source_result.status}')
    print(f'  Model loaded: {source_result.model is not None}')
except Exception as e:
    print(f'Stage 2 source validation: {e}')
print('=' * 80)


VALIDATION: DATASET & CHECKPOINT PROVENANCE
[2026-08-13 20:03:42] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
Dataset root: /kaggle/input/competitions/rsna-pneumonia-detection-challenge

STAGE 1 RESUME VALIDATION (Optimizer State Required)
[2026-08-13 20:03:42] [INFO] [medvision.models.trainer:160] - AUTO-RESUME: no canonical checkpoint found at /kaggle/working/medvision_outputs/checkpoints/densenet121_stage1_best.keras
Stage 1 checkpoint: densenet121_stage1_best.keras
Stage 1 checkpoint validation: 'NoneType' object has no attribute 'status'

STAGE 2 SOURCE VALIDATION (Model Weights Only, Optimizer NOT Required)
Stage 2 source validation: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'NoneType'


## Stage 2 Numerical Forensic Experiment

This controlled experiment compares Stage 2 fine-tuning behavior under FP32 and mixed-float16 precision.

It must be completed before the production Stage 2 launcher is executed.

The experiment:
- uses the existing validated Stage 1 checkpoint as the Stage 2 source;
- does not retrain Stage 1;
- compares numerical stability;
- records the first bad batch/tensor when instability occurs;
- must not modify the canonical Stage 1 checkpoint.

Run this cell only after Cell 3 has confirmed that the Stage 1 checkpoint is a valid Stage 2 source.

In [5]:
from pathlib import Path

matches = list(Path("/kaggle/input").rglob("densenet121_stage1_best.keras"))

print("Found:", len(matches))

for p in matches:
    print(p)

Found: 1
/kaggle/input/datasets/swastikpandey004/stage1-folder/medvision_stage1_artifacts/checkpoints/densenet121_stage1_best.keras


In [6]:
from pathlib import Path
import shutil

matches = list(Path("/kaggle/input").rglob("densenet121_stage1_best.keras"))

if len(matches) != 1:
    raise RuntimeError(
        f"Expected exactly one Stage 1 checkpoint, found {len(matches)}: {matches}"
    )

source = matches[0]

destination = (
    Path("/kaggle/working/medvision_outputs")
    / "checkpoints"
    / "densenet121_stage1_best.keras"
)

destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, destination)

print("Copied:")
print("Source:", source)
print("Destination:", destination)

Copied:
Source: /kaggle/input/datasets/swastikpandey004/stage1-folder/medvision_stage1_artifacts/checkpoints/densenet121_stage1_best.keras
Destination: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage1_best.keras


In [7]:
from pathlib import Path

checkpoint = Path(
    "/kaggle/working/medvision_outputs/checkpoints/"
    "densenet121_stage1_best.keras"
)

print("Exists:", checkpoint.exists())

if checkpoint.exists():
    print("Size MB:", round(checkpoint.stat().st_size / 1024**2, 2))

Exists: True
Size MB: 33.33


In [8]:
import sys

sys.path.insert(0, "/kaggle/working/MedVision-AI/src")

from medvision.models.trainer import validate_stage2_source_checkpoint

result = validate_stage2_source_checkpoint(
    str(checkpoint),
    "densenet121",
)

print("=" * 70)
print("STAGE 2 SOURCE VALIDATION")
print("=" * 70)
print("Status: VALID")
print("Path:", result.path)
print("Size MB:", round(result.size_bytes / 2**20, 2))
print("=" * 70)

I0000 00:00:1786651502.313577      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786651502.316618      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


STAGE 2 SOURCE VALIDATION
Status: VALID
Path: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage1_best.keras
Size MB: 33.33


In [9]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/kaggle/working/MedVision-AI")

STAGE1_CHECKPOINT = (
    Path("/kaggle/working/medvision_outputs")
    / "checkpoints"
    / "densenet121_stage1_best.keras"
)

print("=" * 75)
print("STARTING STAGE 2 NUMERICAL FORENSIC EXPERIMENT")
print("=" * 75)

print("Repository:", REPO)
print("Stage 1 source:", STAGE1_CHECKPOINT)
print("Checkpoint exists:", STAGE1_CHECKPOINT.exists())

if not STAGE1_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Stage 1 checkpoint not found: {STAGE1_CHECKPOINT}"
    )

subprocess.run(
    [
        sys.executable,
        "scripts/train.py",
        "--mode",
        "full",
        "--stage",
        "stage2_forensic",
        "--epochs",
        "1",
        "--batch-size",
        "32",
    ],
    cwd=REPO,
    check=True,
)

STARTING STAGE 2 NUMERICAL FORENSIC EXPERIMENT
Repository: /kaggle/working/MedVision-AI
Stage 1 source: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage1_best.keras
Checkpoint exists: True
[2026-08-13 19:20:13] [INFO] [medvision.train_script:324] - ===========================================================================
[2026-08-13 19:20:13] [INFO] [medvision.train_script:325] - MedVision-AI Controlled Cloud Training Engine
[2026-08-13 19:20:13] [INFO] [medvision.train_script:326] - ===========================================================================
[2026-08-13 19:20:13] [INFO] [medvision.train_script:327] - Execution Mode       : full
[2026-08-13 19:20:13] [INFO] [medvision.train_script:328] - Model Architecture   : densenet121
[2026-08-13 19:20:13] [INFO] [medvision.train_script:329] - Target Stage         : stage2_forensic
[2026-08-13 19:20:13] [INFO] [medvision.train_script:330] - Max Epochs per Stage : 1
[2026-08-13 19:20:13] [INFO] [medvision.train_scrip

I0000 00:00:1786648814.190701     182 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13590 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786648814.193000     182 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13654 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[2026-08-13 19:20:14] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 19:20:14] [INFO] [medvision.train_script:161] - REAL_RSNA_DATASET = YES
[2026-08-13 19:20:14] [INFO] [medvision.train_script:163] - Dataset Root Resolved: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 19:20:14] [INFO] [medvision.train_script:164] - Labels File Resolved : /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv
[2026-08-13 19:20:14] [INFO] [medvision.train_script:426] - Full mode selected: Resolving RSNA dataset root & data pipelines...
[2026-08-13 19:20:14] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 19:20:14] [INFO] [medvision.train_script:446] - TFRecord shards not found at '/kaggle/working/MedVision-AI/data/processed/tfrecords'. Building real RSNA DICOM datasets direc

I0000 00:00:1786648895.706894     211 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


[2026-08-13 19:21:36] [INFO] [medvision.models.trainer:638] - [BATCH 01/10] loss=0.4670 | weights_finite=True | opt_vars_finite=True | metrics_clean=True
 1/10 ━━━━━━━━━━━━━━━━━━━━ 5:31 37s/step - accuracy: 0.8125 - f1_score: 0.4000 - loss: 0.4670 - pr_auc: 0.5786 - precision: 0.3333 - recall: 0.5000 - roc_auc: 0.7143 - specificity: 0.8571[2026-08-13 19:21:36] [INFO] [medvision.models.trainer:638] - [BATCH 02/10] loss=0.4634 | weights_finite=True | opt_vars_finite=True | metrics_clean=True
 2/10 ━━━━━━━━━━━━━━━━━━━━ 2s 253ms/step - accuracy: 0.8281 - f1_score: 0.5077 - loss: 0.4652 - pr_auc: 0.5207 - precision: 0.4167 - recall: 0.6500 - roc_auc: 0.7627 - specificity: 0.8545[2026-08-13 19:21:36] [INFO] [medvision.models.trainer:638] - [BATCH 03/10] loss=0.5506 | weights_finite=True | opt_vars_finite=True | metrics_clean=True
 3/10 ━━━━━━━━━━━━━━━━━━━━ 2s 363ms/step - accuracy: 0.8160 - f1_score: 0.5051 - loss: 0.4937 - pr_auc: 0.4986 - precision: 0.4293 - recall: 0.6185 - roc_auc: 0.770

CompletedProcess(args=['/usr/bin/python3', 'scripts/train.py', '--mode', 'full', '--stage', 'stage2_forensic', '--epochs', '1', '--batch-size', '32'], returncode=0)

## Stage 1 Launcher (Optional)

**Only run if a valid Stage 1 checkpoint does NOT already exist.**

Stage 1:
- Trains only the DenseNet121 classification head
- Freezes backbone (ImageNet pretrained weights)
- Outputs: `densenet121_stage1_best.keras` with full optimizer state
- Duration: ~4-8 hours on GPU

If a valid checkpoint exists, Stage 2 will auto-resume from it via `--auto-resume`.


In [ ]:
# OPTIONAL: Stage 1 launcher
# Only run if no valid Stage 1 checkpoint exists.
# Duration: ~4-8 hours on GPU

import subprocess
import sys

ENABLE_STAGE1 = False  # Set to True only if needed

if ENABLE_STAGE1:
    os.chdir(str(REPO))
    print('=' * 80)
    print('STARTING STAGE 1: DenseNet121 Head Training')
    print('=' * 80)
    cmd = [
        sys.executable,
        'scripts/train.py',
        '--mode', 'full',
        '--stage', 'stage1',
        '--batch-size', '32',
        '--epochs', '3',
        '--mixed-precision',
    ]
    result = subprocess.run(cmd)
    sys.exit(result.returncode)
else:
    print('Stage 1 is DISABLED (optional).')
    print('Set ENABLE_STAGE1 = True above only if no valid Stage 1 checkpoint exists.')


## Stage 2 Production Launcher (ACTIVE)

### Workflow
1. Auto-resumes from valid Stage 1 checkpoint (via `--auto-resume`)
2. Unfreezes top 20 DenseNet121 layers
3. Keeps BatchNorm frozen
4. Fine-tunes with optimizer: Adam (lr=1e-5, clipnorm=1.0)
5. Outputs: `densenet121_stage2_best.keras` (model weights only, no optimizer state)
6. Duration: ~2-4 hours on GPU for 3 epochs

### Source Checkpoint Behavior

- Searches for `densenet121_stage1_best.keras`
- Validates the checkpoint as a Stage 2 model-weight source
- Stage 2 does NOT require the Stage 1 optimizer state
- Compatible model weights are loaded directly
- A fresh Stage 2 optimizer is used for fine-tuning
- Stage 1 is NOT rerun when the Stage 2 source checkpoint is valid


In [10]:
# PRODUCTION: Stage 2 Fine-Tuning Launcher (ACTIVE)
# This is the main training path.
# It auto-resumes from valid Stage 1 checkpoint.

# ============================================================
# STAGE 2 PRODUCTION FINE-TUNING
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

REPO = Path("/kaggle/working/MedVision-AI")

STAGE1_CHECKPOINT = (
    Path("/kaggle/working/medvision_outputs")
    / "checkpoints"
    / "densenet121_stage1_best.keras"
)

STAGE2_CHECKPOINT = (
    Path("/kaggle/working/medvision_outputs")
    / "checkpoints"
    / "densenet121_stage2_best.keras"
)

print("=" * 80)
print("STARTING STAGE 2: DenseNet121 Fine-Tuning (Production)")
print("=" * 80)

print("Repository:", REPO)
print("Stage 1 source checkpoint:", STAGE1_CHECKPOINT)
print("Stage 1 checkpoint exists:", STAGE1_CHECKPOINT.exists())

if not REPO.exists():
    raise FileNotFoundError(
        f"Repository not found: {REPO}"
    )

if not STAGE1_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Stage 1 checkpoint not found: {STAGE1_CHECKPOINT}"
    )

os.chdir(str(REPO))

print("Working directory:", Path.cwd())
print("=" * 80)

cmd = [
    sys.executable,
    "scripts/train.py",
    "--mode", "full",
    "--stage", "stage2",
    "--epochs", "3",
    "--batch-size", "32",
    "--mixed-precision",
    "--auto-resume",
]

print("Running command:")
print(" ".join(cmd))
print("=" * 80)

result = subprocess.run(cmd)

print("=" * 80)
print("STAGE 2 PROCESS RETURN CODE:", result.returncode)
print("=" * 80)

sys.exit(result.returncode)

STARTING STAGE 2: DenseNet121 Fine-Tuning (Production)
Repository: /kaggle/working/MedVision-AI
Stage 1 source checkpoint: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage1_best.keras
Stage 1 checkpoint exists: True
Working directory: /kaggle/working/MedVision-AI
Running command:
/usr/bin/python3 scripts/train.py --mode full --stage stage2 --epochs 3 --batch-size 32 --mixed-precision --auto-resume
[2026-08-13 19:23:39] [INFO] [medvision.train_script:324] - ===========================================================================
[2026-08-13 19:23:39] [INFO] [medvision.train_script:325] - MedVision-AI Controlled Cloud Training Engine
[2026-08-13 19:23:39] [INFO] [medvision.train_script:326] - ===========================================================================
[2026-08-13 19:23:39] [INFO] [medvision.train_script:327] - Execution Mode       : full
[2026-08-13 19:23:39] [INFO] [medvision.train_script:328] - Model Architecture   : densenet121
[2026-08-13 19:23:39] [

I0000 00:00:1786649024.635408     417 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13590 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786649024.637621     417 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13654 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[2026-08-13 19:23:44] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 19:23:44] [INFO] [medvision.train_script:161] - REAL_RSNA_DATASET = YES
[2026-08-13 19:23:44] [INFO] [medvision.train_script:163] - Dataset Root Resolved: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 19:23:44] [INFO] [medvision.train_script:164] - Labels File Resolved : /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv
[2026-08-13 19:23:44] [INFO] [medvision.train_script:426] - Full mode selected: Resolving RSNA dataset root & data pipelines...
[2026-08-13 19:23:44] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 19:23:44] [INFO] [medvision.train_script:446] - TFRecord shards not found at '/kaggle/working/MedVision-AI/data/processed/tfrecords'. Building real RSNA DICOM datasets direc

[2026-08-13 19:24:22] [INFO] [medvision.models.trainer:993] - INPUT       : dtype=<dtype: 'float32'> | shape=(16, 224, 224, 3) | min=0.0000 | max=1.0000 | mean=0.5477 | finite=True
[2026-08-13 19:24:22] [INFO] [medvision.models.trainer:996] - LABELS      : dtype=<dtype: 'float32'> | unique=[0.0, 1.0] | pos=4 | neg=12 | finite=True
[2026-08-13 19:24:22] [INFO] [medvision.models.trainer:999] - PREDICTIONS : dtype=<dtype: 'float32'> | shape=(16, 1) | min=0.0571 | max=0.9736 | finite=True
[2026-08-13 19:24:22] [INFO] [medvision.models.trainer:1002] - RAW BCE     : value=1.1744 | finite=True | dtype=float32
[2026-08-13 19:24:22] [INFO] [medvision.models.trainer:1005] - WEIGHTED BCE: value=1.3770 | finite=True | dtype=float32
[2026-08-13 19:24:22] [INFO] [medvision.models.trainer:1008] - GRADIENTS   : norm=23.7212 | None_count=0 | range=[-2.5469e+00, 2.9590e+00] | finite=True
[2026-08-13 19:24:22] [INFO] [medvision.models.trainer:1011] - OPTIMIZER   : class=LossScaleOptimizer | lr=9.99999974

I0000 00:00:1786649121.805039    1005 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13590 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786649121.807513    1005 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13654 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[2026-08-13 19:25:21] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 19:25:21] [INFO] [medvision.train_script:161] - REAL_RSNA_DATASET = YES
[2026-08-13 19:25:21] [INFO] [medvision.train_script:163] - Dataset Root Resolved: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 19:25:21] [INFO] [medvision.train_script:164] - Labels File Resolved : /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv
[2026-08-13 19:25:21] [INFO] [medvision.train_script:426] - Full mode selected: Resolving RSNA dataset root & data pipelines...
[2026-08-13 19:25:21] [INFO] [medvision.data:36] - Dataset root auto-detected at: /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[2026-08-13 19:25:21] [INFO] [medvision.train_script:446] - TFRecord shards not found at '/kaggle/working/MedVision-AI/data/processed/tfrecords'. Building real RSNA DICOM datasets direc

I0000 00:00:1786649170.284414    1034 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


[2026-08-13 19:26:10] [INFO] [medvision.models.trainer:638] - [BATCH 01/584] loss=0.5951 | weights_finite=True | opt_vars_finite=True | metrics_clean=True
  1/584 ━━━━━━━━━━━━━━━━━━━━ 6:05:14 38s/step - accuracy: 0.6875 - f1_score: 0.6154 - loss: 0.5951 - pr_auc: 0.4703 - precision: 0.5000 - recall: 0.8000 - roc_auc: 0.7818 - specificity: 0.6364[2026-08-13 19:26:11] [INFO] [medvision.models.trainer:638] - [BATCH 02/584] loss=0.6278 | weights_finite=True | opt_vars_finite=True | metrics_clean=True
  2/584 ━━━━━━━━━━━━━━━━━━━━ 2:44 283ms/step - accuracy: 0.6406 - f1_score: 0.5251 - loss: 0.6114 - pr_auc: 0.3964 - precision: 0.4167 - recall: 0.7125 - roc_auc: 0.7386 - specificity: 0.6098 [2026-08-13 19:26:11] [INFO] [medvision.models.trainer:638] - [BATCH 03/584] loss=0.5711 | weights_finite=True | opt_vars_finite=True | metrics_clean=True
  3/584 ━━━━━━━━━━━━━━━━━━━━ 3:40 380ms/step - accuracy: 0.6493 - f1_score: 0.5265 - loss: 0.5980 - pr_auc: 0.4053 - precision: 0.4206 - recall: 0.7058

SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [9]:
from pathlib import Path
from medvision.models.trainer import verify_checkpoint_persistence

stage2_ckpt = CHECKPOINTS / 'densenet121_stage2_best.keras'

print('=' * 80)
print('STAGE 2 CHECKPOINT VERIFICATION')
print('=' * 80)

if stage2_ckpt.exists():
    print(f'Stage 2 checkpoint found: {stage2_ckpt}')
    print(f'  Size: {stage2_ckpt.stat().st_size / 1e6:.2f} MB')
    
    # Verify persistence
    try:
        verification_result = verify_checkpoint_persistence(str(stage2_ckpt))
        print(f'  Persistence check: {verification_result.status}')
        print(f'  Reason: {verification_result.reason}')
    except Exception as e:
        print(f'  Verification error: {e}')
else:
    print('Stage 2 checkpoint NOT found.')
    print(f'Expected path: {stage2_ckpt}')
    print('Ensure Stage 2 training completed successfully.')
print('=' * 80)


STAGE 2 CHECKPOINT VERIFICATION
Stage 2 checkpoint NOT found.
Expected path: /kaggle/working/medvision_outputs/checkpoints/densenet121_stage2_best.keras
Ensure Stage 2 training completed successfully.


In [10]:
from pathlib import Path

stage2_ckpt = Path(
    "/kaggle/working/medvision_outputs/checkpoints/"
    "densenet121_stage2_best.keras"
)

print("=" * 70)
print("STAGE 2 FINAL CHECKPOINT")
print("=" * 70)
print("Exists:", stage2_ckpt.exists())

if stage2_ckpt.exists():
    print("Size MB:", round(stage2_ckpt.stat().st_size / 1024**2, 2))
    print("Path:", stage2_ckpt)

STAGE 2 FINAL CHECKPOINT
Exists: False


In [13]:
from pathlib import Path

p = Path(
    "/kaggle/working/medvision_outputs/checkpoints/"
    "densenet121_stage2_best.keras"
)

print(p.exists())

False


In [11]:
import os
from pathlib import Path

print('=' * 80)
print('ARTIFACT INVENTORY')
print('=' * 80)

runtime_dir = Path('/kaggle/working/medvision_outputs')
if runtime_dir.exists():
    print(f'Runtime artifacts directory: {runtime_dir}\n')
    
    for root, dirs, files in os.walk(runtime_dir):
        level = root.replace(str(runtime_dir), '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        sub_indent = ' ' * 2 * (level + 1)
        for file in sorted(files):
            file_path = Path(root) / file
            size_mb = file_path.stat().st_size / 1e6
            print(f'{sub_indent}{file} ({size_mb:.2f} MB)')
else:
    print(f'Runtime directory not found: {runtime_dir}')

print('\n' + '=' * 80)


ARTIFACT INVENTORY
Runtime artifacts directory: /kaggle/working/medvision_outputs

medvision_outputs/
  checkpoints/
    densenet121_stage1_best.keras (34.95 MB)



In [12]:
import shutil
from pathlib import Path

runtime_dir = Path('/kaggle/working/medvision_outputs')
zip_path = Path('/kaggle/working/medvision_stage1_stage2_artifacts.zip')

print('=' * 80)
print('CREATE FINAL ARTIFACT ZIP')
print('=' * 80)

if runtime_dir.exists():
    print(f'Zipping {runtime_dir} -> {zip_path}...')
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', runtime_dir.parent, runtime_dir.name)
    zip_size_mb = zip_path.stat().st_size / 1e6
    print(f'Created: {zip_path} ({zip_size_mb:.2f} MB)')
else:
    print(f'No runtime directory found at {runtime_dir}')
print('=' * 80)


CREATE FINAL ARTIFACT ZIP
Zipping /kaggle/working/medvision_outputs -> /kaggle/working/medvision_stage1_stage2_artifacts.zip...
Created: /kaggle/working/medvision_stage1_stage2_artifacts.zip (31.39 MB)


In [1]:
from IPython.display import FileLink
from pathlib import Path

zip_path = Path('/kaggle/working/medvision_stage1_stage2_artifacts.zip')

if zip_path.exists():
    print('Download your artifacts:')
    display(FileLink(str(zip_path)))
else:
    print('No ZIP file found. Run the artifact packaging cell above first.')


No ZIP file found. Run the artifact packaging cell above first.
